# Haya Skin Analyzer — YOLOv8 Training v2
**Run on Kaggle with T4 GPU** (Settings → Accelerator → GPU T4 x1)

Datasets: Acne · Dark Circles · Wrinkles
Target time: ~1 hour

In [ ]:
# Cell 1 — Install
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

In [ ]:
# Cell 2 — Config
ROBOFLOW_API_KEY = "Dej9oDfQTdZXUPl6xeQe"
EPOCHS     = 60
MODEL_SIZE = "yolov8n"
IMG_SIZE   = 416
BATCH      = 32

In [ ]:
# Cell 3 — Download datasets, fix class names, and merge
from roboflow import Roboflow
import os, shutil, yaml, re
from pathlib import Path

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

datasets_to_download = [
    ("xzesoxee",                 "acne-detection-9z3qf", 3),
    ("skin-condition-detection",  "dark-circles-19mqw",   1),
    ("wrinkle-old-bucket",        "wrinkle-rkcym",       17),
]

# ── Class name normalizer ────────────────────────────────────────
# Maps any messy dataset class name → our clean canonical name
def normalize_class(name: str) -> str | None:
    n = str(name).lower().strip()
    # Acne
    if any(k in n for k in ["acne", "pimple", "spot", "blackhead", "whitehead", "comedone", "blemish"]):
        return "acne"
    # Dark circles
    if any(k in n for k in ["dark", "circle", "puffy", "eye bag", "eyebag", "periorbital"]):
        return "dark_circles"
    # Wrinkles
    if any(k in n for k in ["wrinkle", "fine line", "fineline", "crow", "forehead"]):
        return "wrinkles"
    # Pigmentation
    if any(k in n for k in ["pigment", "melasma", "freckle", "dark spot", "hyperpig"]):
        return "pigmentation"
    # Pure numbers or unknown → skip
    if re.fullmatch(r'\d+', n):
        return None
    return None   # unknown class — skip it

# ── Download ─────────────────────────────────────────────────────
downloaded = []
for workspace, project_name, version_num in datasets_to_download:
    try:
        print(f"Downloading {project_name}...")
        ds = rf.workspace(workspace).project(project_name).version(version_num).download("yolov8")
        downloaded.append(ds.location)
        print(f"  OK: {ds.location}")
    except Exception as e:
        print(f"  FAILED {project_name}: {e}")

if not downloaded:
    raise RuntimeError("No datasets downloaded — check API key.")

# ── Build class registry ─────────────────────────────────────────
# For each dataset: read its classes, build a remap {old_id → canonical_name}
CANONICAL = ["acne", "dark_circles", "wrinkles", "pigmentation"]
MERGED = Path("/kaggle/working/merged")
for split in ("train", "valid", "test"):
    (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)

for ds_path in downloaded:
    ds_path = Path(ds_path)
    yaml_file = next(ds_path.glob("*.yaml"), None)
    if not yaml_file:
        print(f"  WARNING: no yaml in {ds_path}, skipping")
        continue

    with open(yaml_file) as f:
        dy = yaml.safe_load(f)
    raw_names = dy.get("names", [])

    # Map old class id → new canonical id (or None to drop)
    id_remap = {}
    for old_id, raw_name in enumerate(raw_names):
        canonical = normalize_class(raw_name)
        if canonical and canonical in CANONICAL:
            id_remap[old_id] = CANONICAL.index(canonical)
        else:
            print(f"  Dropping unknown class: '{raw_name}'")

    print(f"  {ds_path.name}: {raw_names} → remap={id_remap}")

    # Copy images and rewrite labels with new class ids
    for split in ("train", "valid", "test"):
        for img in (ds_path / split / "images").glob("*"):
            shutil.copy(img, MERGED / split / "images" / img.name)

        for lbl in (ds_path / split / "labels").glob("*.txt"):
            new_lines = []
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    old_id = int(parts[0])
                    if old_id in id_remap:
                        parts[0] = str(id_remap[old_id])
                        new_lines.append(" ".join(parts))
                    # else: drop this annotation (unknown class)
            if new_lines:
                dst = MERGED / split / "labels" / lbl.name
                with open(dst, "w") as f:
                    f.write("\n".join(new_lines))

# ── Write final data.yaml ────────────────────────────────────────
DATA_YAML = str(MERGED / "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.dump({
        "path":  str(MERGED),
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(CANONICAL),
        "names": CANONICAL,
    }, f)

print("\nFinal class names:", CANONICAL)
for split in ("train", "valid", "test"):
    n = len(list((MERGED / split / "images").glob("*")))
    print(f"  {split}: {n} images")
print("\nDataset ready.")

In [ ]:
# Cell 4 — Train
from ultralytics import YOLO

model = YOLO(f"{MODEL_SIZE}.pt")

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH,
    patience    = 20,
    save        = True,
    save_period = 5,
    project     = "/kaggle/working",
    name        = "haya_skin_v2",
    exist_ok    = True,
    device      = 0,
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    warmup_epochs= 3,
    weight_decay = 0.0005,
    augment  = True,
    mosaic   = 1.0,
    mixup    = 0.1,
    fliplr   = 0.5,
    degrees  = 10.0,
    hsv_s    = 0.5,
    hsv_v    = 0.3,
    val     = True,
    plots   = True,
    verbose = False,
)

print("Training complete.")

In [ ]:
# Cell 5 — Save to output root so Kaggle can download it
import shutil, glob, os
from ultralytics import YOLO

candidates = glob.glob("/kaggle/working/**/best.pt", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/working/**/last.pt", recursive=True)
if not candidates:
    raise FileNotFoundError("No model found — check training logs above.")

# Copy to output root
shutil.copy(candidates[0], "/kaggle/working/best.pt")
size_mb = os.path.getsize("/kaggle/working/best.pt") / 1024 / 1024
print(f"best.pt saved ({size_mb:.1f} MB)")

# Validate per class
m = YOLO("/kaggle/working/best.pt")
metrics = m.val(data=DATA_YAML, verbose=False)
print(f"\nmAP50:     {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print(f"Classes:   {m.names}")
print("\nPer-class mAP50:")
for i, (name, ap) in enumerate(zip(m.names.values(), metrics.box.ap50)):
    print(f"  {name}: {ap:.3f}")

print("\n" + "="*50)
print("NEXT STEPS:")
print("1. Click 'Save Version' (top right in Kaggle)")
print("2. Choose 'Save & Run All (Commit)'")
print("3. Wait for it to finish (~1 hour)")
print("4. Go to Output tab → download best.pt")
print("5. Place at: backend/app/models/skin_model.pt")
print("="*50)